In [ ]:
import os, sys
import numpy as np
import pickle as pk
from random import randint
from itertools import product, chain
import scipy.interpolate as itp
from multiprocessing import Pool, Process

sys.path.append('/home/nishant/lab/MFB/scripts')
sys.path.append('/home/nishant/lab/MFB/steps')
sys.path.append('/home/nishant/lab/scripts')
from analysis import *
from peaks import *
from misc import *

## MCell analysis

In [ ]:
dataPath = "/media/nishant/4tb/output/MFB/findPr/"
resultPath = "/home/nishant/lab/MFB/results/findPr/"
fname = "ca.dat" 

dirs = [a for a in getDirs(dataPath, sstr='nVDCC_') if "nVDCC_" in a]
dirs = np.array(dirs)[:]
print(dirs.shape)
#for i,d in enumerate(dirs): print(i, d)

### Averaging Calcium Data

In [ ]:
for d in dirs:
    try:
        avg_dat(dataPath, resultPath, d, 'cb_mol.dat', cores=40)
        avg_dat(dataPath, resultPath, d, 'pmca&leak_ca_flux.dat', cores=40)
        print(f'{d} done!')
    except:
        print(f'problem in {d}')

In [ ]:
fname = "ca.dat" 
for dir in tq(dirs[:], desc=f'AVG ({fname}):'):
    #avg_dat(dataPath, resultPath, dir, fname, cores=0)
    #'''
    try:
        avg_dat(dataPath, resultPath, dir, fname, cores=40)
        print(dir)
    except:
        print(f'Error in {dir}')
    #'''

In [ ]:
fname = "ca.dat" 
jobInfo = list(product([dataPath], [resultPath], tempdirs[:], [fname]))

p = Pool(30)
p.starmap(avg_dat, jobInfo)
p.close()

### Generate AZ Calcium Concentration

In [ ]:
#CaConc(resultPath, tempdirs[1], step=10)

arg = list(product([resultPath], dirs[:]))
p = Pool(40)
p.starmap(CaConc, arg);
p.close()

## Plot MCell data

In [ ]:
tempdirs = ['nVDCC_9_dVDCC_120_nAZ_7']
def plotCa(dir):
    CaGlobal = np.genfromtxt(os.path.join(resultPath,dir,'ca.dat'), unpack=True)
    CaAZ = np.genfromtxt(os.path.join(resultPath,dir,'CaConc.dat'), unpack=True)

    f, ax = plt.subplots(2, sharex=True, figsize=(15,8))
    f.subplots_adjust(hspace=0.0)

    ax[0].plot(CaGlobal[0]*1000, CaGlobal[1]*2.024e-4)
    ax[0].set_ylabel('Global Ca ($\mu$M)')
    ax[0].set_title(dir)

    for i in range(1,30):#len(CaAZ)+1):
        ax[1].plot(CaAZ[0]*1000, CaAZ[i], label="AZ_{:d}".format(i))

    ax[1].set_xlim(0, 5)
    ax[1].set_ylabel("$[Ca]$ ($\mu$M)")
    ax[1].legend(bbox_to_anchor=(1, 1.2), loc='upper left', ncol=2)
    #ax[1].set_ylim(0,0.4)
    plt.show()
    
for dir in tempdirs:
    plotCa(dir)

## AZ simulation in STEPS

In [ ]:
from MFB_model import *
mdl, sim, r = get_MFB_model()

## Testing a single trail with STEPS

In [ ]:
CaFile = resultPath + 'nVDCC_8_dVDCC_100_nAZ_28/CaConc.dat'
CaData = np.genfromtxt(CaFile, unpack=True)

T = np.arange(CaData[0][0], CaData[0][-1], 1e-5)
CaInterp = itp.interp1d(CaData[0], CaData[1])
CaInterpData = CaInterp(T)

resCa, resAZ, vesRel = simAZ(T, CaInterpData, sim, r)

nFig = 3
figure, ax = plt.subplots(nFig, figsize=(15, 4*nFig), sharex=True)
figure.subplots_adjust(hspace=0.1)
labelfontsize = 13

ax[0].plot(T, resCa/cytVolVal*1.66*1e-3, label='Ca')
ax[0].set_ylabel(r'Ca ($\mu M$)', fontsize=labelfontsize)

for mol in azMolName[:18]:
    i = azMolName.index(mol)
    ax[1].plot(T, resAZ[:,i], label=mol)
ax[1].set_ylabel('AZ states (num)', fontsize=labelfontsize)
ax[1].legend(bbox_to_anchor=(1, 1), loc='upper left', ncol=2)

vesRelSync  = np.sum(vesRel[:,:3], axis=1)
vesRelAsync = np.sum(vesRel[:,3:-1], axis=1)
vesRelSpont = vesRel[:,-1]
vesRelTot   = np.sum(vesRel, axis=1)

ax[2].plot(T, vesRelSync, label='sync')
ax[2].plot(T, vesRelAsync, label='async')
ax[2].plot(T, vesRelSpont, label='spont')
ax[2].plot(T, vesRelTot, label='total')
    
ax[2].set_ylabel('Vesicles released', fontsize=labelfontsize)
ax[2].legend()

peaks = detect_peaks(vesRelTot, edge='rising', show=False)
ax[2].plot(T[peaks], vesRelTot[peaks], '+', mfc=None, mec='r', mew=2, ms=15)
#print(T[peaks])

plt.show()

In [ ]:
def runAZTrials(CaData, RRPs):
    vesData = []
    for RRP in RRPs:
        resAZ, vesRel = simAZ(CaData[0], CaData[1], sim, r, RRP=RRP)
        vesRelTot = np.sum(vesRel, axis=1)
        pks = detect_peaks(vesRelTot, edge='rising', show=False)
        vesData.append(list(CaData[0,pks]))
    
    return vesData

In [ ]:
def getVesRel(dir, RRPs, resultPath, fname='CaConc.dat', trials=2000):
    nAZ = int(dir.split('_')[-1])
    rrps = len(RRPs)
    
    CaFile = os.path.join(resultPath, dir, fname)
    CaData = np.genfromtxt(CaFile, unpack=True) # in uM

    p = Pool(38)
    vesRelTimes = []
    for iCa in tq(range(1,nAZ+1), desc=dir):
        
        info = product([CaData[(0,iCa),:]], [RRPs]*trials)
        #print(list(info))
        vesRelTime = p.starmap(runAZTrials, info)
        vesRelTime = [[vesRelTime[i][j] for i in range(trials)] for j in range(rrps)]
        vesRelTimes.append(vesRelTime)

    p.close()
    p.join()
    
    vesRelTimes = [[list(chain(*[vesRelTimes[i][j][k] for i in range(nAZ)])) for k in range(trials)] for j in range(rrps)]
    vesData = {}
    for RRP,v in zip(RRPs, vesRelTimes):
        vesData.update({str(RRP): list(v)})

    return vesData

dir = 'nVDCC_9_dVDCC_80_nAZ_7'
#vesData = getVesRel(dir, RRPs=range(5,41,5), resultPath=resultPath, trials=10)

In [ ]:
dir = 'nVDCC_9_dVDCC_80_nAZ_7'
vesData = getVesRel(dir, RRPs=range(5,41,5), resultPath=resultPath, trials=10)

#niceprint(vesData)
for k,v in vesData.items():
    print(k)
    print(v,)

In [ ]:
vdccs = list(range(1,10))
dvdccs = [60, 70, 90, 110]
nazs = list(range(8,30)) # 7 has been mising... gotta do those 7s !!!
tempdirs = []
for vdcc, dvdcc, naz in product(vdccs, dvdccs, nazs):
    tempdirs.append(f'nVDCC_{vdcc}_dVDCC_{dvdcc}_nAZ_{naz}')
#print(tempdirs)

for dir in tq(tempdirs):
    try:
        vesData = getVesRel(dir, RRPs=range(5,41,5), resultPath=resultPath, trials=2000)
        with open(os.path.join(resultPath, dir, 'vesData.dat'),"wb") as outfile:
            pk.dump(vesData, outfile)
    except:
        print(f'Error in {dir}')

In [ ]:
for dir in tq(dirs):
    #print(dir)
    try:
        vesData = getVesRel(dir, RRPs=range(5,41,5), resultPath=resultPath, trials=1000)
        with open(os.path.join(resultPath, dir, 'vesData.dat'),"wb") as outfile:
            pk.dump(vesData, outfile)
    except:
        print(f'Error in {dir}')
        
    try:
        vesData = PrStat(dir, resultPath=resultPath, resample=1000)
    except:
        print(f'Error in Pr calc. for {dir}')

In [ ]:
p = Pool()
res = p.map(getVesData, tempdirs)
p.close()
p.join()

## Get Pr stats

In [ ]:
for dir in tq(dirs[:]):
    try:
        vesData = PrStat(dir, resultPath=resultPath, resample=1000)
    except:
        print(f'Error in {dir}')

## Extra